# Telecom Customer Churn Prediction

End-to-end **supervised** **classification** workflow for the Telco customer dataset. Intended for **Google Colab** (data path: `/content/sample_data/Telco_Customer_Churn.csv`).

**Supervised vs unsupervised:** This project is **supervised ML** because each customer row includes a known outcome label (`Churn`: Yes/No). The models learn to map features → that label. **Unsupervised** ML would apply if there were no churn column—e.g. only clustering customers into groups (K-means) or dimensionality reduction (PCA) without using a target variable.

**Note:** Churn prediction is a classification problem (target: Yes/No). *R² is a regression metric*; this notebook uses **F1-score** and **ROC-AUC** alongside accuracy in the comparison table—these play the same role as extra generalization checks beyond raw accuracy.

**Google Colab tips**

- Upload `Telco_Customer_Churn.csv` to **Files → sample_data** (or adjust `COLAB_PATH` in the next section) so it matches `/content/sample_data/Telco_Customer_Churn.csv`.
- Optional XGBoost benchmark: run `!pip install xgboost -q` in a code cell once if the import fails.

## 1. Environment & imports

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import skew

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

try:
    from xgboost import XGBClassifier

    HAS_XGB = True
except ImportError:
    HAS_XGB = False

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)


## 2. Import data

Colab default: `sample_data`. Falls back to `Telco_Customer_Churn.csv` in the current directory for local Jupyter.

In [ ]:
COLAB_PATH = "/content/sample_data/Telco_Customer_Churn.csv"
LOCAL_CANDIDATES = ["Telco_Customer_Churn.csv", "sample_data/Telco_Customer_Churn.csv"]

if os.path.isfile(COLAB_PATH):
    DATA_PATH = COLAB_PATH
else:
    DATA_PATH = next((p for p in LOCAL_CANDIDATES if os.path.isfile(p)), COLAB_PATH)

df = pd.read_csv(DATA_PATH)
print("Loaded from:", DATA_PATH)
print("Shape:", df.shape)
df.head()


## 3. Exploratory Data Analysis (EDA)

In [ ]:
df.info()
df.describe(include="all").T


In [ ]:
churn_map = {"Yes": 1, "No": 0}
df["Churn_binary"] = df["Churn"].map(churn_map)
print(df["Churn"].value_counts(normalize=True))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["Churn"].value_counts().plot(kind="bar", ax=ax[0], color=["steelblue", "coral"], edgecolor="black")
ax[0].set_title("Churn class counts")
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=0)
df["Churn"].value_counts(normalize=True).plot(kind="bar", ax=ax[1], color=["steelblue", "coral"], edgecolor="black")
ax[1].set_title("Churn class proportion")
ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plot_df = df.copy()
plot_df["TotalCharges_num"] = pd.to_numeric(plot_df["TotalCharges"], errors="coerce")
num_plot = ["tenure", "MonthlyCharges", "TotalCharges_num", "SeniorCitizen"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()
for i, c in enumerate(num_plot):
    sns.histplot(plot_df[c].dropna(), kde=True, ax=axes[i], color="teal")
    axes[i].set_title(f"Distribution: {c}")
plt.tight_layout()
plt.show()

for c in ["tenure", "MonthlyCharges", "TotalCharges_num"]:
    s = plot_df[c].dropna()
    print(c, "skew=", round(skew(s), 3))


In [ ]:
cat_cols = [
    c
    for c in df.columns
    if c not in ["customerID", "Churn", "Churn_binary", "TotalCharges"] and df[c].dtype == "object"
]

n = len(cat_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = np.array(axes).reshape(-1)
for idx, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df["Churn"], normalize="index")
    ct.plot(kind="bar", ax=axes[idx], stacked=False, color=["steelblue", "coral"], edgecolor="black")
    axes[idx].set_title(f"Churn rate by {col}")
    axes[idx].legend(title="Churn")
    axes[idx].tick_params(axis="x", rotation=45)
for j in range(len(cat_cols), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
corr_df = plot_df[num_plot + ["Churn_binary"]].dropna()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Correlation heatmap (numeric features + Churn)")
plt.tight_layout()
plt.show()


### EDA summary (refine after you review plots)

- **Class imbalance:** Churn "Yes" is usually the minority class; prefer F1 / ROC-AUC, not only accuracy.
- **Contract & tenure:** Longer contracts and higher tenure typically associate with lower churn.
- **Fiber / electronic check:** Often higher-churn segments in Telco benchmarks.
- **TotalCharges:** Aligns with tenure and monthly charges; may be skewed—consider transforms after imputation.

## 4. Missing values & outliers

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0]
print("Explicit NaN counts:\n", missing if len(missing) else "None")

df["TotalCharges_clean"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("TotalCharges non-numeric -> NaN:", df["TotalCharges_clean"].isna().sum())
print(df.loc[df["TotalCharges_clean"].isna(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]].head(15))


In [ ]:
mask_na = df["TotalCharges_clean"].isna()
df.loc[mask_na, "TotalCharges_clean"] = df.loc[mask_na, "tenure"] * df.loc[mask_na, "MonthlyCharges"]
still_na = df["TotalCharges_clean"].isna()
if still_na.any():
    df.loc[still_na, "TotalCharges_clean"] = df["TotalCharges_clean"].median()


def iqr_outlier_mask(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    return (s < lo) | (s > hi)


for col in ["tenure", "MonthlyCharges", "TotalCharges_clean"]:
    m = iqr_outlier_mask(df[col])
    print(col, "IQR outliers:", m.sum(), f"({100 * m.mean():.2f}%)")


## 5. Feature engineering & preprocessing

- Drop `customerID`.
- Use cleaned numeric `TotalCharges`.
- **Stratified** `train_test_split` on `Churn`.
- **ColumnTransformer:** median imputer + `StandardScaler` for numerics; `OneHotEncoder` for categoricals.

In [ ]:
X = df.drop(columns=["customerID", "Churn", "Churn_binary", "TotalCharges"])
X["TotalCharges"] = df["TotalCharges_clean"]

y = df["Churn_binary"]

numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [c for c in X.columns if c not in numeric_features]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", y_train.mean().round(3))


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)
print("Transformed shape:", X_train_p.shape)


In [ ]:
raw_num = X_train[numeric_features].copy()
for c in ["MonthlyCharges", "TotalCharges"]:
    print(c, "skew (train)=", round(skew(raw_num[c]), 3))


## 6. Model building — multiple classifiers

**Run order:** Use **Runtime → Run all** after a restart, or run sections **1 → 5** before this cell. The code below rebuilds `X_train_p` / `X_test_p` automatically if they are missing (as long as `df` is already loaded).

**Note:** *Linear Regression* is not appropriate for a binary class label. The standard linear baseline for churn is **Logistic Regression** (included below). Optional course trick: regressing on `Churn_binary` with `LinearRegression` yields an R² on a 0/1 target, but it is **not** recommended for model selection—this notebook uses proper classifiers only.

In [ ]:
# If X_train_p is missing (skipped section 5 or Runtime → Restart), rebuild from df.
if "X_train_p" not in globals():
    if "df" not in globals():
        raise NameError(
            "Load the dataset first (section 2). This cell can rebuild preprocessing only when `df` exists."
        )
    if "Churn_binary" not in df.columns:
        df["Churn_binary"] = df["Churn"].map({"Yes": 1, "No": 0})
    if "TotalCharges_clean" not in df.columns:
        df["TotalCharges_clean"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        _na = df["TotalCharges_clean"].isna()
        df.loc[_na, "TotalCharges_clean"] = df.loc[_na, "tenure"] * df.loc[_na, "MonthlyCharges"]
        _na2 = df["TotalCharges_clean"].isna()
        if _na2.any():
            df.loc[_na2, "TotalCharges_clean"] = df["TotalCharges_clean"].median()
    X = df.drop(columns=["customerID", "Churn", "Churn_binary", "TotalCharges"])
    X["TotalCharges"] = df["TotalCharges_clean"]
    y = df["Churn_binary"]
    numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
    categorical_features = [c for c in X.columns if c not in numeric_features]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )
    X_train_p = preprocessor.fit_transform(X_train)
    X_test_p = preprocessor.transform(X_test)
    print("Rebuilt X_train_p / X_test_p (run section 5 before this cell for one clean top-to-bottom pass).")


def eval_classifier(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    pred_tr = model.predict(Xtr)
    pred_te = model.predict(Xte)
    tr_acc = accuracy_score(ytr, pred_tr)
    te_acc = accuracy_score(yte, pred_te)
    tr_f1 = f1_score(ytr, pred_tr)
    te_f1 = f1_score(yte, pred_te)
    tr_auc = te_auc = np.nan
    if hasattr(model, "predict_proba"):
        proba_tr = model.predict_proba(Xtr)[:, 1]
        proba_te = model.predict_proba(Xte)[:, 1]
        tr_auc = roc_auc_score(ytr, proba_tr)
        te_auc = roc_auc_score(yte, proba_te)
    elif hasattr(model, "decision_function"):
        tr_auc = roc_auc_score(ytr, model.decision_function(Xtr))
        te_auc = roc_auc_score(yte, model.decision_function(Xte))
    overfit = "Y" if (tr_acc - te_acc) > 0.08 or (tr_f1 - te_f1) > 0.08 else "N"
    return {
        "Model": name,
        "Train accuracy": round(tr_acc, 4),
        "Train F1": round(tr_f1, 4),
        "Train ROC-AUC": round(tr_auc, 4) if tr_auc == tr_auc else np.nan,
        "Test accuracy": round(te_acc, 4),
        "Test F1": round(te_f1, 4),
        "Test ROC-AUC": round(te_auc, 4) if te_auc == te_auc else np.nan,
        "Overfitting (Y/N)": overfit,
        "_fitted": model,
    }


models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=8),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "KNN (k=11)": KNeighborsClassifier(n_neighbors=11),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
}

if HAS_XGB:
    models["XGBoost"] = XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        eval_metric="logloss",
        verbosity=0,
    )

voting = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=2000, random_state=42)),
        ("rf", RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)),
        ("gb", GradientBoostingClassifier(random_state=42)),
    ],
    voting="soft",
    n_jobs=-1,
)
models["Voting (LR+RF+GB)"] = voting

results = []
fitted = {}
for name, clf in models.items():
    print("Training:", name, "...")
    row = eval_classifier(name, clf, X_train_p, X_test_p, y_train, y_test)
    fitted[name] = row.pop("_fitted")
    results.append(row)

results_df = pd.DataFrame(results)
results_df


## 7. Model evaluation — comparison table

For classification, **F1** and **ROC-AUC** replace **R²** from regression templates. The **champion model** is chosen by **Test F1** (focus on the minority churn class). ROC-AUC remains in the table for context.

In [ ]:
display_cols = [
    "Model",
    "Train accuracy",
    "Train F1",
    "Test accuracy",
    "Test F1",
    "Test ROC-AUC",
    "Overfitting (Y/N)",
]
comparison = results_df[display_cols].sort_values("Test F1", ascending=False).reset_index(drop=True)
comparison


In [ ]:
best_name = comparison.iloc[0]["Model"]
print("Best model (by Test F1):", best_name)
best_model = fitted[best_name]
print(classification_report(y_test, best_model.predict(X_test_p), target_names=["No churn", "Churn"]))
cm = confusion_matrix(y_test, best_model.predict(X_test_p))
plt.figure(figsize=(4, 3))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"],
)
plt.title(f"Confusion matrix — {best_name}")
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.tight_layout()
plt.show()


## 8. Export artifacts for Streamlit

Saves the **Test F1** champion (`best_model` / `best_name` from section 7), the fitted `preprocessor`, and default values for the Streamlit form. Files go under `artifacts/` (create this folder in Colab if needed).

For a **local clone** without running the full notebook, you can run `python train_export.py` instead (trains Logistic Regression to match the usual champion).

In [ ]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(preprocessor, artifacts_dir / "preprocessor.joblib")
joblib.dump(best_model, artifacts_dir / "model.joblib")
(artifacts_dir / "model_name.txt").write_text(best_name, encoding="utf-8")

form_defaults = {}
for col in X_train.columns:
    s = X_train[col]
    if pd.api.types.is_numeric_dtype(s):
        form_defaults[col] = float(s.median())
    else:
        form_defaults[col] = str(s.mode(dropna=True).iloc[0])

joblib.dump(form_defaults, artifacts_dir / "form_defaults.joblib")

print("Saved to:", artifacts_dir.resolve())
for p in sorted(artifacts_dir.iterdir()):
    if p.is_file():
        print(" -", p.name, f"({p.stat().st_size} bytes)")


## 9. GitHub & Streamlit Community Cloud

The repo includes `app.py` (Streamlit UI), `inference_utils.py`, `requirements.txt`, and **`DEPLOY.md`** with step-by-step commands: generate `artifacts/`, `git push` to GitHub, then create an app on [share.streamlit.io](https://share.streamlit.io) with **Main file path = `app.py`**.

**Colab:** download the `artifacts/` folder from the Files pane and add it to your local repo before pushing, or clone the repo in Colab, run this notebook + export cell, then `git push` from Colab (requires git auth).